In [ ]:
import pymysql
import csv

# -----------------------------------
# 1. DB 연결
# -----------------------------------
conn = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="testuser",    # local user
    password="1234",      # Yisup2026@
    database="DBPr01"     ############ CHANGE  Before make database 
)

cur = conn.cursor()

# -----------------------------------
# 2. CSV 파일 경로 / 테이블 이름
# -----------------------------------
fn = '/Users/dh/projects/Lectures/Dataenginering/703cdf01-ff09-4b86-b017-6e8d87b11fd2.csv'
table_name = 'ga0_raw'

# -----------------------------------
# 3. CSV header 읽기
# -----------------------------------
with open(fn, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)

print("CSV header count:", len(header))
print("CSV header:", header)

# -----------------------------------
# 4. 컬럼명 정리
#    공백 -> _
#    소문자 변환
# -----------------------------------
clean_header = []
for col in header:
    col = col.strip().lower().replace(' ', '_')
    clean_header.append(col)

print("Clean header:", clean_header)

# -----------------------------------
# 5. 기존 테이블 삭제
# -----------------------------------
cur.execute(f"DROP TABLE IF EXISTS {table_name};")
conn.commit()

# -----------------------------------
# 6. CREATE TABLE 자동 생성
#    모든 컬럼을 일단 VARCHAR(300)으로 생성
# -----------------------------------
column_defs = []
for col in clean_header:
    column_defs.append(f"`{col}` VARCHAR(300)")

create_sql = f"""
CREATE TABLE {table_name} (
    id_column INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
    {', '.join(column_defs)}
);
"""

print("=== CREATE SQL ===")
print(create_sql)

cur.execute(create_sql)
conn.commit()

# -----------------------------------
# 7. INSERT SQL 자동 생성
# -----------------------------------
column_names_sql = ", ".join([f"`{col}`" for col in clean_header])
placeholders = ", ".join(["%s"] * len(clean_header))

insert_sql = f"""
INSERT INTO {table_name} (
    {column_names_sql}
) VALUES (
    {placeholders}
);
"""

print("=== INSERT SQL ===")
print(insert_sql)

# -----------------------------------
# 8. CSV 데이터 읽기
# -----------------------------------
data_rows = []

with open(fn, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)  # header skip

    for row in reader:
        data_rows.append(tuple(row))

print("Rows to insert:", len(data_rows))

# -----------------------------------
# 9. executemany로 한꺼번에 insert
# -----------------------------------
cur.executemany(insert_sql, data_rows)
conn.commit()

print("INSERT 완료")

# -----------------------------------
# 10. 확인
# -----------------------------------
cur.execute(f"SELECT COUNT(*) FROM {table_name};")
print("Rows in table:", cur.fetchone()[0])

# -----------------------------------
# 11. 종료
# -----------------------------------
cur.close()
conn.close()